# Train All Experiments (6 configurations x 3 seeds = 18 runs)

| Model | Scheme | Heads |
|---|---|---|
| A | single-task | species (8) |
| B | single-task | freshness (3) |
| C | flat 24-class | combined (24) |
| D-EW / D-UW / D-DWA | multi-task | 8 + 3, one weighting strategy each |

Reads the committed `split_manifest.csv` -- the split is never redrawn here. Checkpoints and history save to Drive, and already-checkpointed runs are skipped, so an interrupted session resumes by re-running this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'
CHECKPOINT_DIR = '/content/drive/MyDrive/fish-freshness-mtl/checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/fish-freshness-mtl/results'
CURVES_DIR = f'{RESULTS_DIR}/figures/curves'

import os
for d in ['/content/data', CHECKPOINT_DIR, RESULTS_DIR, CURVES_DIR]:
    os.makedirs(d, exist_ok=True)

!unzip -q -n "$DATASET_ZIP" -d /content/data

In [ ]:
import sys
sys.path.append('/content/repo/04_Src')

import pandas as pd
from split_utils import load_split, verify_split
from train import train_single_task, train_flat24, train_multitask, TrainConfig
from plotting import plot_single_task_curves, plot_multitask_curves

split_df = pd.read_csv('/content/repo/02_Manifests/split_manifest.csv')
verify_split(split_df)  # re-checks the committed split before spending GPU time
train_df, val_df, test_df = load_split('/content/repo/02_Manifests/split_manifest.csv')
len(train_df), len(val_df), len(test_df)

In [ ]:
SEEDS = [42, 43, 44]

EXPERIMENTS = [
    {'name': 'ModelA_species',   'kind': 'single', 'task': 'species'},
    {'name': 'ModelB_freshness', 'kind': 'single', 'task': 'freshness'},
    {'name': 'ModelC_flat24',    'kind': 'flat24'},
    {'name': 'ModelD_EW',        'kind': 'mtl', 'strategy': 'EW'},
    {'name': 'ModelD_UW',        'kind': 'mtl', 'strategy': 'UW'},
    {'name': 'ModelD_DWA',       'kind': 'mtl', 'strategy': 'DWA'},
]
print(f'{len(EXPERIMENTS) * len(SEEDS)} runs total')

Model A and B are listed first on purpose: running them before the rest gives the difficulty gap between the two tasks, which is what the asymmetry argument rests on.

In [ ]:
import json
import matplotlib.pyplot as plt

cfg = TrainConfig()
common = dict(train_df=train_df, val_df=val_df, dataset_root=DATASET_ROOT, cfg=cfg)

for exp in EXPERIMENTS:
    for seed in SEEDS:
        run_name = f"{exp['name']}_seed{seed}"
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'{run_name}.pt')
        if os.path.exists(ckpt_path):
            print(f'[{run_name}] checkpoint exists, skipping')
            continue
        print(f'[{run_name}] starting...')

        if exp['kind'] == 'single':
            result = train_single_task(exp['task'], checkpoint_path=ckpt_path, seed=seed,
                                       run_name=run_name, **common)
        elif exp['kind'] == 'flat24':
            result = train_flat24(checkpoint_path=ckpt_path, seed=seed,
                                  run_name=run_name, **common)
        else:
            result = train_multitask(exp['strategy'], checkpoint_path=ckpt_path, seed=seed,
                                     run_name=run_name, **common)

        with open(os.path.join(RESULTS_DIR, f'{run_name}_history.json'), 'w') as f:
            json.dump(result['history'], f)

        plot = plot_single_task_curves if exp['kind'] == 'single' else plot_multitask_curves
        fig = plot(result['history'], run_name, save_path=os.path.join(CURVES_DIR, f'{run_name}.png'))
        plt.close(fig)

## Training summary

Rebuilt from what is on Drive rather than from this session's variables, so runs completed in an earlier session are still included.

In [ ]:
import torch

rows = []
for exp in EXPERIMENTS:
    for seed in SEEDS:
        run_name = f"{exp['name']}_seed{seed}"
        history_path = os.path.join(RESULTS_DIR, f'{run_name}_history.json')
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'{run_name}.pt')

        if os.path.exists(history_path):
            with open(history_path) as f:
                history = json.load(f)
            key = 'val_f1_mean' if 'val_f1_mean' in history[0] else 'val_f1_macro'
            rows.append({'run_name': run_name, 'model': exp['name'], 'seed': seed,
                         'best_val_f1': max(h[key] for h in history),
                         'epochs_trained': len(history), 'source': 'complete'})
        elif os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            rows.append({'run_name': run_name, 'model': exp['name'], 'seed': seed,
                         'best_val_f1': ckpt.get('val_f1_macro', ckpt.get('val_f1_mean')),
                         'epochs_trained': ckpt['epoch'] + 1, 'source': 'checkpoint_only'})

summary_df = pd.DataFrame(rows)
summary_df.to_csv(os.path.join(RESULTS_DIR, 'training_summary.csv'), index=False)
summary_df

Everything above is written to Drive. `/content/repo` is discarded when the runtime recycles; to put these results in the repository, download them and commit from a machine with push access.